In [28]:
import pandas as pd 
import sqlite3
from sqlalchemy import create_engine

In [29]:
df = pd.read_csv('online_retail_II.csv')

In [30]:
print(df.shape)
print(df.dtypes)

(1067371, 8)
Invoice         object
StockCode       object
Description     object
Quantity         int64
InvoiceDate     object
Price          float64
Customer ID    float64
Country         object
dtype: object


In [31]:
print(df.head())

  Invoice StockCode                          Description  Quantity  \
0  489434     85048  15CM CHRISTMAS GLASS BALL 20 LIGHTS        12   
1  489434    79323P                   PINK CHERRY LIGHTS        12   
2  489434    79323W                  WHITE CHERRY LIGHTS        12   
3  489434     22041         RECORD FRAME 7" SINGLE SIZE         48   
4  489434     21232       STRAWBERRY CERAMIC TRINKET BOX        24   

           InvoiceDate  Price  Customer ID         Country  
0  2009-12-01 07:45:00   6.95      13085.0  United Kingdom  
1  2009-12-01 07:45:00   6.75      13085.0  United Kingdom  
2  2009-12-01 07:45:00   6.75      13085.0  United Kingdom  
3  2009-12-01 07:45:00   2.10      13085.0  United Kingdom  
4  2009-12-01 07:45:00   1.25      13085.0  United Kingdom  


In [41]:
db_path = "retail.db"
engine = create_engine(f"sqlite:///{db_path}")
df.to_sql("transactions_raw", engine, if_exists="replace", index=False)

print(f"Written {df.shape[0]} rows and {df.shape[1]} columns to database at {db_path}")

Written 1067371 rows and 8 columns to database at retail.db


In [42]:
from sqlalchemy import text

In [51]:
create_view = """
CREATE VIEW IF NOT EXISTS transactions_clean AS
SELECT *
FROM transactions_raw
WHERE Quantity > 0
  AND Price > 0
  AND Invoice NOT LIKE 'C%'
  AND "Customer ID" IS NOT NULL;
"""

In [52]:
with engine.begin() as conn:
    conn.execute(text(create_view))       

In [53]:
with engine.connect() as conn:
    tables = conn.execute(text("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;")).fetchall()
    views  = conn.execute(text("SELECT name FROM sqlite_master WHERE type='view' ORDER BY name;")).fetchall()
    raw_count = conn.execute(text("SELECT COUNT(*) FROM transactions_raw;")).scalar()
    clean_count = conn.execute(text("SELECT COUNT(*) FROM transactions_clean;")).scalar()
    sample = conn.execute(text('SELECT Invoice, StockCode, Description, Quantity, InvoiceDate, Price, "Customer ID", Country FROM transactions_clean LIMIT 5;')).fetchall()

print("tables:", tables)
print("views :", views)
print(f"Raw table row count: {raw_count}")
print(f"Cleaned view row count: {clean_count}")
sample

tables: [('transactions_raw',), ('transcractions',)]
views : [('transactions_clean',), ('transcations_clean',)]
Raw table row count: 1067371
Cleaned view row count: 805549


[('489434', '85048', '15CM CHRISTMAS GLASS BALL 20 LIGHTS', 12, '2009-12-01 07:45:00', 6.95, 13085.0, 'United Kingdom'),
 ('489434', '79323P', 'PINK CHERRY LIGHTS', 12, '2009-12-01 07:45:00', 6.75, 13085.0, 'United Kingdom'),
 ('489434', '79323W', ' WHITE CHERRY LIGHTS', 12, '2009-12-01 07:45:00', 6.75, 13085.0, 'United Kingdom'),
 ('489434', '22041', 'RECORD FRAME 7" SINGLE SIZE ', 48, '2009-12-01 07:45:00', 2.1, 13085.0, 'United Kingdom'),
 ('489434', '21232', 'STRAWBERRY CERAMIC TRINKET BOX', 24, '2009-12-01 07:45:00', 1.25, 13085.0, 'United Kingdom')]

In [54]:
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS transcractions;"))
print("Dropped table transcractions (if it existed).")

Dropped table transcractions (if it existed).


In [55]:
with engine.begin() as conn:
    conn.execute(text("DROP VIEW IF EXISTS transcations_clean;"))
print("Dropped view transcations_clean (if it existed).")

Dropped view transcations_clean (if it existed).


In [56]:
with engine.connect() as conn:
    tables = conn.execute(text("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;")).fetchall()
    views  = conn.execute(text("SELECT name FROM sqlite_master WHERE type='view' ORDER BY name;")).fetchall()

print("tables:", tables)
print("views :", views)


tables: [('transactions_raw',)]
views : [('transactions_clean',)]
